In [ ]:
import cv2
import numpy as np
import os

def apply_exposure_adjustment(img, gamma_values=None, linear_factors=None, base_filename="image"):
    exposure_images = []
    filenames = []

    # Gamma correction if gamma values are provided
    if gamma_values is not None:
        for idx, gamma in enumerate(gamma_values, start=1):
            gamma_table = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
            exposure_images.append(cv2.LUT(img, gamma_table))
            filenames.append(f"{base_filename}_gamma{idx}.png")  # Gamma-corrected image

    # Original image
    exposure_images.append(img)
    filenames.append(f"{base_filename}_original.png")

    # Linear exposure adjustment if linear factors are provided
    if linear_factors is not None:
        for idx, factor in enumerate(linear_factors, start=1):
            exposure_images.append(np.clip(img * factor, 0, 255).astype(np.uint8))
            filenames.append(f"{base_filename}_linear{idx}.png")  # Linear adjusted image

    return exposure_images, filenames

def save_images(images, filenames, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for img, filename in zip(images, filenames):
        save_path = os.path.join(output_dir, filename)
        cv2.imwrite(save_path, img)

# Parameters
input_image_path = '1509_1.png'
output_dir = 'output'
base_filename = '1509'

# Load image
image = cv2.imread(input_image_path)

# Define gamma values and linear factors for different exposure adjustments
gamma_values = [2.5, 4.5]  # Example gamma values for gamma correction
linear_factors = [2.0, 4.0]  # Example factors for linear exposure adjustment

# Adjust exposure and get filenames
exposure_images, filenames = apply_exposure_adjustment(image, gamma_values=gamma_values, linear_factors=linear_factors, base_filename=base_filename)

# Save adjusted images with specific filenames
save_images(exposure_images, filenames, output_dir)



error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


In [ ]:
import cv2 as cv
import numpy as np

# Define filenames and exposure times
filenames = [
    "output/1509_gamma1.png",
    "output/1509_gamma2.png",
    "output/1509_linear1.png",
    "output/1509_linear2.png",
    "output/1509_original.png"
]

exposure_times = np.array([15.0, 2.5, 0.25, 0.0333, 0.01], dtype=np.float32)

# Load images into a list
img_list = [cv.imread(fn) for fn in filenames]

# Check if images loaded successfully
if any(img is None for img in img_list):
    print("Error: One or more images could not be loaded.")
else:
    # Merge exposures to HDR image
    merge_debevec = cv.createMergeDebevec()
    hdr_debevec = merge_debevec.process(img_list, times=exposure_times.copy())

    merge_robertson = cv.createMergeRobertson()
    hdr_robertson = merge_robertson.process(img_list, times=exposure_times.copy())

    # Tonemap HDR image
    tonemap = cv.createTonemap(gamma=2.2)
    res_debevec = tonemap.process(hdr_debevec.copy())

    # Exposure fusion using Mertens
    merge_mertens = cv.createMergeMertens()
    res_mertens = merge_mertens.process(img_list)

    # Convert datatype to 8-bit and save
    res_debevec_8bit = np.clip(res_debevec * 255, 0, 255).astype('uint8')
    res_robertson_8bit = np.clip(hdr_robertson * 255, 0, 255).astype('uint8')
    res_mertens_8bit = np.clip(res_mertens * 255, 0, 255).astype('uint8')


    cv.imwrite("finalinal.jpg", res_mertens_8bit)
    print("HDR processing completed. Results saved.")

HDR processing completed. Results saved.


C:\Users\Admin\AppData\Local\Temp\ipykernel_22960\4007462591.py:38: RuntimeWarning: invalid value encountered in cast
  res_debevec_8bit = np.clip(res_debevec * 255, 0, 255).astype('uint8')
